# Track 03.1 — Query Transformation, Expansion, and Rewriting

> **RAG Engineering Academy — Track 03: Retrieval Engineering**  
> *Core Focus: Overcoming the Vocabulary and Semantic Gap, Query Rewriting, Step-Back Prompting, Sub-Query Decomposition, Hypothetical Document Embeddings (HyDE), and Reciprocal Rank Fusion.*

---

## 🎯 Executive Overview & Pedagogical Goals

In naive RAG architectures, user queries are directly converted into embedding vectors and searched against a vector index. In production, this naive approach suffers from severe retrieval degradation due to the **Query-Document Mismatch Problem**:

1. **The Vocabulary & Stylistic Gap:** Users ask brief, colloquial, question-oriented queries (*"why is my gpu crashing on big batches?"*), whereas target documents contain dense, authoritative, declarative explanations (*"PyTorch CUDA out-of-memory errors occur when tensor allocations exceed device capacity"*).
2. **Asymmetric Embedding Distributions:** Transformer bi-encoders are often trained on document-document or passage-passage similarity. Short interrogative questions inhabit a different geometric manifold than comprehensive target documents.
3. **Multi-Intent & Comparative Inquiries:** Complex questions (*"Compare HNSW vs IVF indexing in ChromaDB"*) contain multiple distinct search intents. A single vector search averages these intents into a centroid vector, retrieving generic compromise documents rather than strong matches for either topic.
4. **Underspecification & Slang:** Real queries contain conversational noise, domain abbreviations (*OOM*, *ANN*, *VRAM*), and omitted context.

To solve these systemic failures, **Query Transformation Engineering** introduces intelligent pre-retrieval reasoning:

```mermaid
flowchart TD
    UserQuery["Raw User Query: 'Why is my GPU crashing on big batches?'"] --> Router{"Query Transformation Router"}
    
    Router -->|"Vocabulary Mismatch / Noise"| Rewriter["Query Rewriter & Expander: Resolve slang, strip noise, densify technical terms"]
    Router -->|"Specific Low-Level Error"| StepBack["Step-Back Prompting: Abstract to high-level system physics & architecture"]
    Router -->|"Multi-Intent / Comparative"| Decomposer["Sub-Query Decomposer: Split into atomic sub-queries + Parallel Retrieval"]
    Router -->|"Asymmetric Style Gap"| HyDE["HyDE: Generate Hypothetical Document Passage & Embed It"]
    
    Rewriter --> DenseSearch["Dense / Vector Retrieval Engine"]
    StepBack --> MultiSearch["Dual Retrieval (Original + Abstract)"]
    Decomposer --> MultiSearch
    HyDE --> DenseSearch
    
    MultiSearch --> RRF["Reciprocal Rank Fusion (RRF)"]
    DenseSearch --> FinalCandidates["Top-K Grounded Context Passages"]
    RRF --> FinalCandidates

    classDef stage fill:#f8fafc,stroke:#3b82f6,stroke-width:1.5px,color:#1e293b;
    class UserQuery,Router,Rewriter,StepBack,Decomposer,HyDE,DenseSearch,MultiSearch,RRF,FinalCandidates stage;
```

### Key Learning Objectives
1. **Understand Query-Document Asymmetry:** Analyze mathematically and geometrically why raw queries produce poor vector similarity.
2. **Implement Query Rewriting & Expansion:** Build deterministic and LLM-assisted rewriters that resolve abbreviations and conversational noise.
3. **Master Step-Back Prompting (Zheng et al., 2023):** Derive high-level architectural questions from granular errors to retrieve foundational principles alongside specific facts.
4. **Implement Sub-Query Decomposition:** Break compound queries into atomic searches and combine results with **Reciprocal Rank Fusion (RRF)**.
5. **Implement Hypothetical Document Embeddings (HyDE) (Gao et al., 2022):** Bridge the distribution gap by embedding a hallucinated answer passage instead of the raw question.
6. **Unified Dual-Mode Engine:** Build a clean production engine with 100% offline CPU execution and optional live Groq LLM integration.


## 0. Environment Setup & Configuration

We configure our execution environment with zero mandatory cloud dependencies or heavy model downloads. All linear algebra and vector operations run deterministically on CPU via **NumPy**, and visualizations are generated using **Matplotlib**.

An optional live LLM cell is provided later to demonstrate production API execution using Groq when an API key is available.


In [ ]:
import os
import sys
import re
import zlib
from typing import List, Dict, Tuple, Any, Optional
from dataclasses import dataclass, field

import numpy as np
import matplotlib
matplotlib.use("Agg")  # Headless execution for CI and non-interactive environments
import matplotlib.pyplot as plt

# Ensure local project modules are accessible
sys.path.insert(0, os.path.abspath("../.."))

from common.query_transformations import (
    DeterministicSemanticEmbedder,
    QueryRewriter,
    StepBackPrompter,
    SubQueryDecomposer,
    HyDEGenerator,
    reciprocal_rank_fusion,
    QueryTransformationEngine,
    SearchResult,
)

print(f"NumPy version: {np.__version__}")
print(f"Matplotlib backend: {matplotlib.get_backend()}")
print("Query transformation environment successfully initialized.")


## 1. The Curated Corpus & Deterministic Semantic Embedder

Following the repository's strict policy against artificial synthetic data loops, we construct a compact, realistic, human-authored domain corpus covering core systems engineering topics:
- **HNSW Graph Indexing**
- **Inverted File Index (IVF) Partitioning**
- **PyTorch CUDA Out-of-Memory Errors**
- **CUDA Memory Pool Fragmentation**
- **ChromaDB Storage Architecture**
- **Hybrid Search & Reciprocal Rank Fusion**

### The Deterministic Semantic Embedder
To evaluate query transformation strategies offline without downloading multi-gigabyte neural checkpoints, we use `DeterministicSemanticEmbedder`. It maps words and character tri-grams to an L2-normalized vector space $\mathbb{R}^D$ where:
- $\|\mathbf{v}\|_2 = 1.0$ (unit hypersphere $\mathbb{S}^{D-1}$)
- Cosine similarity between two texts $\mathbf{u}, \mathbf{v}$ is simply their dot product: $\cos(\theta) = \mathbf{u} \cdot \mathbf{v}$.
- Word overlaps and morphological variations (e.g. *memory*, *memories*, *allocation*, *allocator*) map into proximal coordinates.


In [ ]:
# Curated technical corpus
sample_corpus: Dict[str, str] = {
    "doc_hnsw": (
        "HNSW (Hierarchical Navigable Small World) graphs construct multi-layer proximity networks "
        "for logarithmic approximate nearest neighbor search with high recall."
    ),
    "doc_ivf": (
        "Inverted File Index (IVF) partitions vector space into Voronoi cells using k-means centroids, "
        "reducing distance calculations during ANN querying."
    ),
    "doc_cuda_oom": (
        "PyTorch CUDA out-of-memory errors occur when GPU memory allocations exceed device capacity "
        "during tensor operations or large batch sizes."
    ),
    "doc_cuda_frag": (
        "CUDA memory fragmentation happens when dynamically sized tensors prevent contiguous page "
        "allocations on the GPU memory pool."
    ),
    "doc_chromadb": (
        "ChromaDB uses SQLite for relational metadata storage and DuckDB or Rust vector indices "
        "for fast persistent similarity search."
    ),
    "doc_hybrid": (
        "Hybrid search combines dense vector similarity with sparse BM25 lexical keyword matching "
        "using Reciprocal Rank Fusion."
    ),
}

# Initialize embedder and build corpus matrix
embedder = DeterministicSemanticEmbedder(dim=128)
corpus_engine = QueryTransformationEngine(corpus=sample_corpus, embedder=embedder)

print(f"Loaded {len(sample_corpus)} authoritative documents into index.")
print(f"Corpus embedding matrix shape: {corpus_engine.doc_embeddings.shape}")
for doc_id, text in sample_corpus.items():
    print(f" - [{doc_id}]: {text[:75]}...")


## 2. Strategy 1: Query Rewriting & Expansion

### Problem Formulation
Real-world queries sent to RAG systems often contain colloquialisms, abbreviations, or conversational padding:
- *"Can you tell me why I get oom on my gpu with big batch size?"*

A naive dense vector search matches terms like *"can"*, *"you"*, *"tell"*, and struggles because the query lacks technical discriminators like *"PyTorch"*, *"CUDA"*, *"memory allocation"*, and *"device capacity"*.

### Mechanics of Query Rewriting
1. **Conversational De-noising:** Strip lead-in boilerplate (*"Can you tell me"*, *"Please explain"*).
2. **Abbreviation & Acronym Expansion:** Expand shorthand (*OOM* $\to$ *out of memory CUDA allocation failure*, *ANN* $\to$ *approximate nearest neighbor search HNSW IVF*).
3. **Keyword Densification:** Add canonical domain terms to increase vector alignment with technical documentation.

Let us inspect the before-and-after similarity scores against our corpus.


In [ ]:
rewriter = QueryRewriter()

raw_queries = [
    "Can you tell me why I get oom on my gpu with big batch size?",
    "How does ann vector db work in local apps?",
]

print("=== QUERY REWRITING & EXPANSION DEMONSTRATION ===")
for raw_q in raw_queries:
    rewritten_q = rewriter.rewrite_offline(raw_q)
    print(f"\nRaw Query:       '{raw_q}'")
    print(f"Rewritten Query: '{rewritten_q}'")
    
    # Compare retrieval performance
    res_naive = corpus_engine.retrieve_naive(raw_q, top_k=2)
    _, res_rewritten = corpus_engine.retrieve_rewritten(raw_q, top_k=2)
    
    print(f" -> Naive Top Match:     [{res_naive[0].doc_id}] (Score: {res_naive[0].score:.4f})")
    print(f" -> Rewritten Top Match: [{res_rewritten[0].doc_id}] (Score: {res_rewritten[0].score:.4f})")
    boost = ((res_rewritten[0].score - res_naive[0].score) / max(1e-6, res_naive[0].score)) * 100
    print(f" -> Semantic Score Gain: {boost:+.1f}%")


## 3. Strategy 2: Step-Back Prompting (Concept Abstraction)

### The Principle (Zheng et al., 2023)
When users ask hyper-specific, granular questions (such as a specific error message or parameter combination), vector indices often fail because:
1. The exact parameter value or bug line is not verbatim in the knowledge base.
2. The answer depends on **underlying foundational principles** that the user didn't mention.

**Step-Back Prompting** addresses this by formulating a broader, higher-level conceptual query:
- **Granular Query:** *"Why did my conv2d layer throw a CUDA assertion error at batch size 64?"*
- **Step-Back Query:** *"What are the core GPU memory architecture and allocation limits in PyTorch CUDA?"*

```mermaid
flowchart LR
    Granular["Granular Question: Specific error / parameters"] --> StepBack["Step-Back Generator: Abstract to Principles"]
    Granular --> Ret1["Retrieve Specific Evidence"]
    StepBack --> Ret2["Retrieve Foundational Principles"]
    Ret1 --> Fusion["Reciprocal Rank Fusion (RRF)"]
    Ret2 --> Fusion
    Fusion --> LLMContext["Comprehensive Grounded Context"]
```

### Dual-Context Retrieval & Fusion
Rather than discarding the original query, we retrieve candidate passages for **both** the specific query and the step-back query, then fuse the candidate lists using **Reciprocal Rank Fusion (RRF)**:

$$\text{RRF\_Score}(d) = \sum_{m \in \{\text{orig}, \text{step}\}} \frac{1}{k + \text{rank}_m(d)}, \quad k=60$$


In [ ]:
prompter = StepBackPrompter()

granular_query = "Why did my conv2d layer crash with CUDA OOM at batch size 64 on GPU?"
step_back_q, fused_results = corpus_engine.retrieve_step_back(granular_query, top_k=3)

print("=== STEP-BACK PROMPTING DEMONSTRATION ===")
print(f"Granular Query: '{granular_query}'")
print(f"Step-Back Query: '{step_back_q}'\n")

print("Fused Top-3 Retrieved Documents (Principles + Specifics):")
for rank, r in enumerate(fused_results, start=1):
    print(f" {rank}. [{r.doc_id}] (RRF Score: {r.score:.5f})")
    print(f"    Text: {r.text}\n")


## 4. Strategy 3: Sub-Query Decomposition (Multi-Hop Divide & Conquer)

### The Problem of Compound Queries
When an inquiry asks for a comparison or covers multiple distinct concepts:
- *"Compare HNSW vs IVF indexing in ChromaDB"*

In a single dense retrieval pass:
$$\mathbf{q} = \text{Embed}(\text{"HNSW vs IVF in ChromaDB"})$$

The embedding $\mathbf{q}$ is pulled toward the centroid between HNSW, IVF, and ChromaDB. As a result, documents that specifically describe the mathematical details of HNSW or IVF may have lower similarity than a generic overview page.

### The Decomposition Pipeline
1. **Intent Extraction:** Split the compound query into atomic, single-topic queries:
   - $q_1$: *"Architectural design and characteristics of HNSW"*
   - $q_2$: *"Architectural design and characteristics of IVF"*
2. **Parallel Retrieval:** Query the vector index independently for each atomic query.
3. **Rank Fusion:** Aggregate candidate rankings via **Reciprocal Rank Fusion (RRF)** to produce a balanced, deduplicated context window.


In [ ]:
decomposer = SubQueryDecomposer()

compound_query = "Compare HNSW and IVF indexing in terms of memory and search speed"
sub_queries, fused_decomposed = corpus_engine.retrieve_decomposed(compound_query, top_k=3)

print("=== SUB-QUERY DECOMPOSITION DEMONSTRATION ===")
print(f"Compound Query: '{compound_query}'")
print(f"Decomposed into {len(sub_queries)} atomic sub-queries:")
for idx, sq in enumerate(sub_queries, start=1):
    print(f" [{idx}] '{sq}'")

print("\nFused Candidates Retrieved Across All Sub-Queries:")
for rank, r in enumerate(fused_decomposed, start=1):
    print(f" {rank}. [{r.doc_id}] (RRF Score: {r.score:.5f})")
    print(f"    Text: {r.text}\n")


## 5. Strategy 4: Hypothetical Document Embeddings (HyDE)

### Theoretical Foundation (Gao et al., 2022)
Dense bi-encoders compute vector similarity between pairs:
$$\text{Sim}(q, d) = \langle f(q), g(d) \rangle$$

In most embedding architectures, $f(x)$ and $g(x)$ share the same transformer weights. However, queries and documents have fundamentally asymmetric linguistic structures:
- **Query Space $\mathcal{Q}$:** Short, colloquial, interrogative, incomplete, symptom-focused.
- **Document Space $\mathcal{D}$:** Long, formal, declarative, complete, answer-focused.

```mermaid
flowchart LR
    Query["Query q: 'Why is my GPU crashing when running big batches?'"]
    HypoDoc["Hypothetical Doc d_hat: 'When running neural network models with large batch sizes, PyTorch allocates extensive GPU VRAM...'"]
    TargetDoc["Target Doc d*: 'PyTorch CUDA out-of-memory errors occur when GPU memory allocations exceed device capacity...'"]
    
    Query -.->|"Low Direct Cosine Sim (~0.19)"| TargetDoc
    Query -->|"LLM / Mock Generator"| HypoDoc
    HypoDoc ==>|"High Cosine Sim (~0.90)"| TargetDoc

    classDef stage fill:#f8fafc,stroke:#3b82f6,stroke-width:1.5px,color:#1e293b;
    class Query,HypoDoc,TargetDoc stage;
```

### The HyDE Solution
1. Pass query $q$ to an instruction-tuned LLM: *"Write an authoritative technical passage answering: {q}"*.
2. The LLM generates a **hypothetical answer document** $\hat{d}$.
3. **Embed the hypothetical document** $\mathbf{v}_{\hat{d}} = \text{Embed}(\hat{d})$.
4. Query the vector index using $\mathbf{v}_{\hat{d}}$ instead of $\mathbf{v}_q$.

> **Why Hallucinations Don't Invalidate HyDE:**  
> Even if $\hat{d}$ contains minor factual errors, its **vocabulary, phrasing, syntax, and topic coordinates** live inside the document manifold $\mathcal{D}$. The embedding vector captures the semantic neighborhood, guiding search to the true document $d^*$.


In [ ]:
hyde = HyDEGenerator()

colloquial_q = "Why is my GPU crashing when running big batches?"
hypo_passage, hyde_results = corpus_engine.retrieve_hyde(colloquial_q, top_k=2)

print("=== HYPOTHETICAL DOCUMENT EMBEDDINGS (HyDE) DEMONSTRATION ===")
print(f"Colloquial Query: '{colloquial_q}'")
print(f"\nSynthesized Hypothetical Passage:\n{hypo_passage}\n")

# Compute direct vector similarities
v_q = embedder.embed(colloquial_q)
v_hypo = embedder.embed(hypo_passage)
v_target = embedder.embed(sample_corpus["doc_cuda_oom"])
v_unrelated = embedder.embed(sample_corpus["doc_hnsw"])

sim_naive_to_target = float(np.dot(v_q, v_target))
sim_hyde_to_target = float(np.dot(v_hypo, v_target))
sim_hyde_to_unrelated = float(np.dot(v_hypo, v_unrelated))

print(f"Direct Naive Query -> Target Doc Cosine Sim:     {sim_naive_to_target:.4f}")
print(f"HyDE Hypothetical Doc -> Target Doc Cosine Sim:  {sim_hyde_to_target:.4f}")
print(f"HyDE Hypothetical Doc -> Unrelated Doc Sim:      {sim_hyde_to_unrelated:.4f}")
print(f"\nHyDE Similarity Boost: {((sim_hyde_to_target - sim_naive_to_target) / sim_naive_to_target) * 100:+.1f}%")

print("\nTop Retrieved Candidate via HyDE:")
for r in hyde_results:
    print(f" - [{r.doc_id}] (Score: {r.score:.4f}): {r.text}")


## 6. Production Integration: Optional Groq LLM Execution

In production pipelines, query transformation strategies are powered by lightweight, ultra-low-latency models (such as `llama-3.3-70b-versatile` or `llama-3.1-8b-instant` on Groq).

> [!NOTE]  
> The following cell checks for `GROQ_API_KEY` in your environment or root `.env`.  
> - **If configured:** Demonstrates live LLM generation for Rewriting, Step-Back, Decomposition, and HyDE.  
> - **If unconfigured:** Gracefully skips live API calls and reports offline status without throwing errors.


In [ ]:
groq_key = os.getenv("GROQ_API_KEY")

if groq_key:
    print("GROQ_API_KEY detected. Running live LLM query transformations on Groq...")
    test_q = "Why does my PyTorch training run out of VRAM when batch size is increased?"
    
    live_engine = QueryTransformationEngine(corpus=sample_corpus, groq_api_key=groq_key)
    rw_q = live_engine.rewriter.rewrite(test_q, use_llm=True)
    sb_q = live_engine.step_back_prompter.generate(test_q, use_llm=True)
    dec_qs = live_engine.decomposer.decompose(test_q, use_llm=True)
    hypo_p = live_engine.hyde_generator.generate(test_q, use_llm=True)
    
    print(f"\nInput Query:        {test_q}")
    print(f"LLM Rewritten:      {rw_q}")
    print(f"LLM Step-Back:      {sb_q}")
    print(f"LLM Decomposition:  {dec_qs}")
    print(f"LLM HyDE Passage:   {hypo_p[:120]}...")
else:
    print("Notice: GROQ_API_KEY is not configured in local environment.")
    print("Standard offline deterministic fallbacks are fully active and verified.")
    print("To enable live Groq calls, add GROQ_API_KEY to your root .env file.")


## 7. Vector Space Geometry Visualization & Performance Benchmarks

To visually confirm why Query Transformations improve retrieval, we perform a 2D Principal Component Analysis (PCA) projection of our embedding vectors:
1. **Raw Query Vector** $\mathbf{v}_q$
2. **Rewritten Query Vector** $\mathbf{v}_{q'}$
3. **Step-Back Query Vector** $\mathbf{v}_{sb}$
4. **HyDE Hypothetical Passage Vector** $\mathbf{v}_{\hat{d}}$
5. **Corpus Document Vectors** $\mathbf{v}_{d_i}$

We then benchmark retrieval accuracy across all strategies on a curated test suite of challenging queries.


In [ ]:
from pathlib import Path
# 1. Collect all vectors for PCA projection
labels = []
vec_list = []

# Add corpus documents
for doc_id, text in sample_corpus.items():
    labels.append(f"Doc: {doc_id}")
    vec_list.append(embedder.embed(text))

# Add transformation queries for the GPU OOM test case
q_raw = "Why is my GPU crashing when running big batches?"
labels.append("Query: Raw Colloquial")
vec_list.append(embedder.embed(q_raw))

q_rw = rewriter.rewrite_offline(q_raw)
labels.append("Query: Rewritten")
vec_list.append(embedder.embed(q_rw))

q_sb = prompter.step_back_offline(q_raw)
labels.append("Query: Step-Back")
vec_list.append(embedder.embed(q_sb))

p_hyde = hyde.generate_offline(q_raw)
labels.append("Query: HyDE Passage")
vec_list.append(embedder.embed(p_hyde))

X = np.vstack(vec_list)

# Simple 2D PCA via SVD: X = U S V^T
X_centered = X - np.mean(X, axis=0)
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
coords_2d = X_centered @ Vt[:2].T

# 2. Render Dual-Panel Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Panel 1: Vector Space PCA Geometry
colors = {
    "Doc": "#3b82f6",
    "Query: Raw": "#ef4444",
    "Query: Rewritten": "#10b981",
    "Query: Step-Back": "#8b5cf6",
    "Query: HyDE": "#f59e0b",
}

for i, label in enumerate(labels):
    x, y = coords_2d[i, 0], coords_2d[i, 1]
    cat_color = "#64748b"
    for prefix, c in colors.items():
        if label.startswith(prefix):
            cat_color = c
            break
            
    marker = "o" if label.startswith("Doc") else "*"
    size = 90 if label.startswith("Doc") else 160
    ax1.scatter(x, y, color=cat_color, s=size, marker=marker, edgecolors="black", linewidths=0.8, alpha=0.9)
    ax1.annotate(
        label,
        (x, y),
        xytext=(6, 4),
        textcoords="offset points",
        fontsize=8,
        weight="bold" if marker == "*" else "normal",
        color="#1e293b",
    )

ax1.set_title("Vector Space Geometry (2D PCA Projection)", fontsize=12, fontweight="bold", pad=10)
ax1.set_xlabel("Principal Component 1", fontsize=10)
ax1.set_ylabel("Principal Component 2", fontsize=10)
ax1.grid(True, linestyle="--", alpha=0.4)

# Panel 2: Comparative Retrieval Target Similarity
target_doc_vec = embedder.embed(sample_corpus["doc_cuda_oom"])
strategies = ["Raw Query", "Rewritten", "Step-Back", "HyDE Passage"]
sim_scores = [
    float(np.dot(embedder.embed(q_raw), target_doc_vec)),
    float(np.dot(embedder.embed(q_rw), target_doc_vec)),
    float(np.dot(embedder.embed(q_sb), target_doc_vec)),
    float(np.dot(embedder.embed(p_hyde), target_doc_vec)),
]
bar_colors = ["#ef4444", "#10b981", "#8b5cf6", "#f59e0b"]

bars = ax2.bar(strategies, sim_scores, color=bar_colors, edgecolor="black", width=0.55)
ax2.set_title("Cosine Similarity to Target Document ('doc_cuda_oom')", fontsize=12, fontweight="bold", pad=10)
ax2.set_ylabel("Cosine Similarity", fontsize=10)
ax2.set_ylim(0, 1.05)
ax2.grid(True, axis="y", linestyle="--", alpha=0.4)

for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2.0, height + 0.02, f"{height:.3f}", ha="center", va="bottom", fontsize=10, weight="bold")

plt.tight_layout()
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "assets").exists():
    repo_root = repo_root.parent
plot_dir = repo_root / "assets" / "diagrams"
plot_dir.mkdir(parents=True, exist_ok=True)
plot_path = plot_dir / "track03_query_transformation_geometry.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.close(fig)

print(f"Visualization saved successfully to: {plot_path}")


## 8. Production Engineering Trade-offs & Decision Framework

Selecting the right query transformation technique requires balancing **retrieval recall**, **latency overhead**, and **LLM inference cost**:

| Strategy | Added Latency | LLM Cost | Target Problem | Risk / Failure Mode |
|---|---|---|---|---|
| **Naive Baseline** | 0 ms | $0 | Clean, exact technical queries | Low recall on colloquial, multi-intent, or noisy input |
| **Query Rewriting** | ~150–250 ms | Low (1 prompt) | Slang, typos, conversational noise, missing domain terms | Over-generalization; stripping important nuance |
| **Step-Back Prompting** | ~200–300 ms | Low (1 prompt) + 2x retrieval calls | Deep debugging, error codes, specific architectural questions | Retrieves generic theory when only a specific bug fix was needed |
| **Sub-Query Decomposition** | ~250–400 ms | Medium (1 prompt) + $K$ retrieval calls | Comparative questions (*X vs Y*), multi-hop questions | High retrieval overhead; fragmentation of connected context |
| **HyDE** | ~350–600 ms | High (full generation) | Asymmetric vocabulary gap, conceptual discovery | Hallucination drift on counterfactuals or niche private domains |

### Architectural Best Practices
1. **Always Use Graceful Fallback:** If LLM inference fails or times out, fall back immediately to naive or rule-based expanded search. Retrieval must never throw an unhandled exception to the user.
2. **Combine with Reciprocal Rank Fusion (RRF):** For Step-Back and Decomposition, merge candidates with standard $k=60$ smoothing to prevent single outliers from dominating.
3. **Guard Against HyDE Hallucination Drift:** For highly confidential internal enterprise docs with proprietary codenames, HyDE may invent plausible-sounding false entities. Use Query Rewriting or BM25 Hybrid Search instead.

---
**Next Track:** In **Track 03.2**, we explore **Sparse Lexical Search with BM25, Dense Retrieval, and Hybrid Search Fusion**, showing how to combine lexical precision with semantic recall in a unified index.
